In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

In [2]:
central_lat = df_mapa['latitude'].mean()
central_lon = df_mapa['longitude'].mean()

m_mapa = folium.Map(location=[central_lat, central_lon], zoom_start=12, tiles='OpenStreetMap')
m_mapa

In [4]:
for index, row in df_mapa.head(5).iterrows():
    popup_text = f"Tipo: {row['tipo']}<br>Valor de Venda: R$ {row['valor_venda']:,}"
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=popup_text
    ).add_to(m_mapa)

m_mapa

In [5]:
m_circle_mapa = folium.Map(location=[central_lat, central_lon], zoom_start=12, tiles='OpenStreetMap')

for index, row in df_mapa.iterrows():
    fill_color = 'blue' if row['cidade'] == 'Nova Iguaçu' else 'orange'
    popup_text = f"Tipo: {row['tipo']}<br>Valor de Venda: R$ {row['valor_venda']:,}<br>Cidade: {row['cidade']}"

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color='black', # Border color
        fill=True,
        fill_color=fill_color,
        fill_opacity=0.7,
        popup=popup_text,
        tooltip='Clique para detalhes'
    ).add_to(m_circle_mapa)

m_circle_mapa

In [7]:
m_cluster_mapa = folium.Map(location=[central_lat, central_lon], zoom_start=12, tiles='OpenStreetMap')

marker_cluster = MarkerCluster().add_to(m_cluster_mapa)

for index, row in df_mapa.iterrows():
    popup_text = f"Tipo: {row['tipo']}<br>Valor de Venda: R$ {row['valor_venda']:,}<br>Cidade: {row['cidade']}"

    icon_color = ''
    if row['tipo'] == 'Casa':
        icon_color = 'green'
    elif row['tipo'] == 'Apartamento':
        icon_color = 'blue'
    else:
        icon_color = 'gray' # Terreno

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=popup_text,
        icon=folium.Icon(color=icon_color)
    ).add_to(marker_cluster)

m_cluster_mapa